# BIL458 Computer Vision — Term Project

**Title:** CancerAI: A Comparative Study of Deep Learning Strategies for Multi-Modal Medical Image Classification

**Author:** Farah Alhasan (21120205709)

**Instructor:** Assoc. Prof. Murat Gezer

> Before submission: *Kernel → Restart & Run All* and confirm the notebook runs without errors.
>
> The full 48-run grid is the study. It stays off in this notebook until `RUN_FULL_STUDY = True`. Training is planned for 26 November–2 December. The cells above that flag audit the data, define the models, and run the unit tests.

This notebook is the experiment record for four separate classification tasks: brain MRI, breast histopathology, skin lesions, and lung CT. The labels are tumour and lesion categories. The project is not a universal cancer detector and not a clinical device.


## 0. Environment and reproducibility

Main question: how do CNN training strategies and data augmentation change macro-F1 across these four modalities?

- H1. Fine-tuned ResNet18 outperforms a custom CNN and a frozen ResNet18.
- H2. Training augmentation improves generalisation, especially on imbalanced data.
- H3. The gain differs by modality.
- H4. Accuracy hides minority-class errors. Macro-F1 and per-class recall expose them.

Seeds for training are 42, 123, and 2026. The split seed is 42 and is not changed by the training seed. Reported scores are the mean and the sample standard deviation across the three runs.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import platform
import torch
from cancerai.training import get_device, seed_everything

SEED = 42
seed_everything(SEED)
print("python", platform.python_version())
print("torch", torch.__version__)
print("device", get_device())
print("project", ROOT)


## 1. Datasets

The train / validation / test membership is written once, to `results/splits/`, and later runs reload it. That is the leakage control this notebook can actually enforce.

- Breast: patient id from the BreakHis folder name. The code rejects a patient that appears in two splits.
- Skin: official DermaMNIST split. Lesion ids are not in the file, so lesion-level independence is not claimed.
- Brain MRI and lung CT: filenames were checked. They do not yield a patient or scan id. Those results are not described as patient-level.


In [ ]:
from cancerai.datasets import audit_datasets, describe, prepare

report = audit_datasets()
print("data root:", report["data_root"])
for name, entry in report["datasets"].items():
    print("\n==", name, "==")
    print("patient_independent:", entry["patient_independent"])
    print(entry["leakage_note"])
    for split, counts in entry["counts"].items():
        print(f"  {split}: {sum(counts.values())}")


In [ ]:
# One unaugmented bundle per modality. The same split file feeds every model.
for name in ("brain", "breast", "skin", "lung"):
    bundle = prepare(name, augment=False)
    print(describe(bundle))
    print()


## 2. Preprocessing and augmentation

Every model sees a 224×224 RGB image normalized with the ImageNet mean and standard deviation, so the comparison is not confounded by a different input pipeline.

Training augmentation depends on the modality:

- Breast: flips, ±20° rotation, mild brightness and contrast. No hue shift, because stain colour carries information.
- Skin: horizontal flip, ±15° rotation, mild brightness, contrast, and saturation. No hue shift.
- Brain MRI: ±10° rotation. No left-right flip.
- Lung CT: ±8° rotation. No flip and no colour change.

Validation and test images are only resized and normalized.


In [ ]:
from torchvision import transforms as T
from cancerai.datasets import build_transform

def names(dataset, train, augment):
    transform = build_transform(dataset, train=train, augment=augment)
    return [type(step).__name__ for step in transform.transforms]

for dataset in ("brain", "breast", "skin", "lung"):
    print(dataset, "train", names(dataset, True, True))
print("eval", names("brain", False, False))

import matplotlib.pyplot as plt
bundle = prepare("brain", augment=False)
fig, axes = plt.subplots(1, 4, figsize=(10, 2.6))
for axis, index in zip(axes, range(4)):
    axis.imshow(bundle.train_ds.raw(index))
    axis.set_title(bundle.classes[int(bundle.train_ds.rows[index]["label"])], fontsize=8)
    axis.axis("off")
fig.suptitle("Brain MRI training examples (no augmentation)")
plt.show()


## 3. Baseline

The baseline is a four-block CNN trained from scratch: convolution, ReLU, max pooling, then dropout and a linear layer. There is no ImageNet initialization.

A convolution reads a local patch,

$$Y(i,j)=\sum_{u=0}^{k-1}\sum_{v=0}^{k-1} X(i+u,j+v)\,K(u,v)$$

and ReLU is $f(x)=\max(0,x)$. One convolutional layer costs about $O(HW C_{in} C_{out} K^2)$ operations. The custom network uses four convolutions. ResNet18 repeats that pattern through a much deeper residual backbone, which is why it has more parameters and a higher cost. Freezing the backbone trains only the new head.


## 4. Proposed method

The loss is weighted cross-entropy. For a one-hot label this is

$$L=-\sum_{c=1}^{C} w_c y_c \log \hat y_c$$

with $w_c = N / (C n_c)$, rescaled so the weights average to 1. Softmax is inside the loss. The network returns logits.

Macro-F1 gives every class the same weight:

$$F1_{macro}=\frac{1}{C}\sum_{c=1}^{C}\frac{2 P_c R_c}{P_c+R_c}$$

The three ResNet18 strategies are: frozen backbone, fine-tuning with augmentation, and fine-tuning without augmentation. ROC-AUC, when it is defined, is one-versus-rest with macro averaging.


In [ ]:
from cancerai.models import build_model, count_params

for name, freeze, pretrained in (
    ("custom_cnn", False, False),
    ("resnet18", True, False),
    ("resnet18", False, False),
):
    model = build_model(name, n_classes=4, freeze_backbone=freeze, pretrained=pretrained)
    total, trainable = count_params(model)
    label = "frozen head only" if freeze else name
    print(f"{label:22}  total {total:10,d}  trainable {trainable:10,d}")


## 5. Training

Shared loop in `cancerai.training.train_model`: AdamW, weight decay 1e-4, cosine decay to 1e-6, batch size 32. The checkpoint is the epoch with the best validation macro-F1. Patience is 3 and the minimum improvement is 0.001. The test set is scored only after that checkpoint is restored.

| Configuration | Epoch cap | Learning rate |
|---|---:|---:|
| Custom CNN | 10 | 1e-3 |
| Frozen ResNet18 | 8 | 1e-3 |
| Fine-tuned ResNet18, with augmentation | 8 | 1e-4 |
| Fine-tuned ResNet18, without augmentation | 8 | 1e-4 |


In [ ]:
RUN_FULL_STUDY = False  # True runs all 48 jobs. Leave False for Restart & Run All during planning.

if RUN_FULL_STUDY:
    from experiments.run_experiments import main
    main([])
else:
    print("Full study not launched.")
    print("Smoke check:  python -m experiments.run_experiments --smoke")
    print("One real run: python -m experiments.run_experiments --dataset brain --config finetune_aug --seed 42")
    print("All 48 runs:  python -m experiments.run_experiments")


## 6. Evaluation (at least 3 seeds, mean ± std)

Primary metric: test macro-F1. Also accuracy, macro precision, macro recall, the confusion matrix, one-vs-rest macro ROC-AUC when every required class is present, training time, and single-image inference time.


In [ ]:
from experiments.summarize import comparison_tables, load_runs

runs = load_runs()
print(comparison_tables(runs))
print("metric files found:", len(runs))


## 7. Ablation

The ablation is `finetune_aug` against `finetune_no_aug`. Both fine-tune ResNet18 with the same learning rate, the same split, and the same seeds. Only the training transform changes. Validation and test stay unaugmented, so a higher training accuracy from augmentation is not counted as a better model unless the test macro-F1 moves.


## 8. Failure analysis

After each non-smoke run, the five highest-confidence test mistakes are written to `results/plots/` and a Grad-CAM figure for the mistakes is written to `results/gradcam/`. For the report, each example should say:

- true label, predicted label, and confidence
- whether the image is low quality, ambiguous, or close to another class
- whether class imbalance could have pushed the decision
- whether Grad-CAM lands on a meaningful region or on background

The Gradio page is for the presentation. It is not evidence that a prediction is correct. Grad-CAM is not a tumour outline.


In [ ]:
from cancerai.metrics import worst_mistakes

# Shape of the failure picker, on a toy prediction vector.
y_true = [0, 1, 1, 0]
y_pred = [0, 0, 1, 1]
probs = [[0.9, 0.1], [0.8, 0.2], [0.4, 0.6], [0.3, 0.7]]
print("highest-confidence mistakes:", worst_mistakes(y_true, y_pred, probs, k=5))


## 9. Unit tests

These tests check the loss formula, the patient split, the augmentation policy, the macro-F1 definition, and the parameter counts. They do not need the image files.


In [ ]:
import unittest

suite = unittest.defaultTestLoader.loadTestsFromName("tests.test_core")
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "Unit tests failed"
print("Tests passed")


## 10. Demo

`python app.py` opens a local page on port 7860. Choose the modality, upload an image, and read the category, the broader group, and the Grad-CAM map.

The broader group is the category probabilities added together with a fixed map. Brain groups are tumor or no tumor, and tumor is not reported as cancer. Breast groups are benign or malignant. Skin groups are cancer-related or not, with actinic keratoses kept in the cancer-related group because that is the dataset convention. Lung groups are malignant or not.

The demo checkpoint is the fine-tuned ResNet18 with augmentation whose validation macro-F1 was best among the finished seeds. Until those runs exist, the page explains which command trains the checkpoint.


## Conclusion

The study compares a from-scratch CNN, a frozen ResNet18, and a fine-tuned ResNet18 on four modalities, and it separates the effect of augmentation from the effect of fine-tuning. Findings belong in this section after the 48 runs: which hypothesis held, where minority classes failed, and where a missing patient id limits the claim. Future work is a replacement collection with real scan identifiers for MRI and CT, and a lesion-level skin split if the HAM10000 metadata can be joined without breaking the DermaMNIST images.
